# 📖 Notebook 3 — Quorum Reads and Writes (N, W, R)

So far we've had one boss (the primary) and one or more followers. But some very
famous databases — **Amazon Dynamo**, **Cassandra**, **Riak** — deliberately have
*no leader at all*. Any node can answer a write. Any node can answer a read.

How do they stay consistent? With a beautiful, almost mathematical trick called a
**quorum**.

## Learning objectives

- Define **N**, **W**, and **R** and explain what each controls.
- Prove to yourself (by simulation) that **W + R > N** guarantees a read sees the latest write.
- Dial consistency up and down and observe the effect.
- Understand why quorum is a tradeoff, not free magic.


## 🛠️ Setup

Pure Python, no Docker needed. Make sure you've run `uv sync` and selected the `.venv`
kernel in VS Code. If the kernel isn't listed, `Cmd+Shift+P` → "Reload Window".


## 1. What is a quorum?

A **quorum** is just a fancy word for *"enough nodes to agree"*.

In leaderless replication we set three numbers up front:

| Symbol | Meaning | Typical value |
|---|---|---|
| **N** | How many replicas hold a copy of each piece of data. | 3 |
| **W** | How many of them must **acknowledge a write** before it's called successful. | 2 |
| **R** | How many of them must **answer a read** before we trust the result. | 2 |

### The magic rule

> **If `W + R > N`, every read is guaranteed to see the latest successful write.**

Why? Because any set of `W` nodes and any set of `R` nodes out of `N` must share at
least one node in common. That common node is guaranteed to have the newest write.
Let's build it and see.


## 2. A leaderless cluster in ~40 lines

Each node stores a value **with a version number (a timestamp)**. When a read comes
back from several nodes, we keep the one with the highest version — that's how we
figure out which copy is "newest".


In [ ]:
import random
import time
from itertools import count
from typing import Optional

from pydantic import BaseModel, Field


class VersionedValue(BaseModel):
    value: str
    version: int


class Node:
    def __init__(self, name: str):
        self.name = name
        self.store: dict[str, VersionedValue] = {}
        self.online = True

    def write(self, key: str, vv: VersionedValue) -> bool:
        if not self.online:
            return False
        existing = self.store.get(key)
        # Only overwrite if this version is newer — avoids a slow delivery clobbering a newer value.
        if existing is None or vv.version > existing.version:
            self.store[key] = vv
        return True

    def read(self, key: str) -> Optional[VersionedValue]:
        if not self.online:
            return None
        return self.store.get(key)


_version_gen = count(start=1)


def new_version() -> int:
    return next(_version_gen)


class QuorumCluster(BaseModel):
    N: int = Field(gt=0)
    W: int = Field(gt=0)
    R: int = Field(gt=0)
    nodes: list[Node] = Field(default_factory=list)

    model_config = {"arbitrary_types_allowed": True}

    def client_write(self, key: str, value: str) -> bool:
        vv = VersionedValue(value=value, version=new_version())
        acks = 0
        # In a real system we'd do this in parallel and with retries. We keep it sequential for clarity.
        shuffled = random.sample(self.nodes, len(self.nodes))
        for node in shuffled:
            if node.write(key, vv):
                acks += 1
            if acks >= self.W:
                return True
        return False  # Could not reach W acks — write failed.

    def client_read(self, key: str) -> Optional[VersionedValue]:
        # Contact EXACTLY R random nodes and wait for all of them to answer.
        # A node that doesn't know the key answers None ("I have no copy").
        # Whichever non-null answer has the highest version wins.
        picked = random.sample(self.nodes, self.R)
        answers = [n.read(key) for n in picked]
        non_null = [a for a in answers if a is not None]
        if not non_null:
            return None
        return max(non_null, key=lambda v: v.version)


## 3. The happy case: `N=3, W=2, R=2`

`W + R = 4 > N = 3`, so the overlap rule kicks in. Every read should see every
successful write, even if one node is down or slow.


In [ ]:
nodes = [Node(f"n{i}") for i in range(3)]
cluster = QuorumCluster(N=3, W=2, R=2, nodes=nodes)

assert cluster.client_write("flag", "green")
# Force the green write onto every node so the demo is deterministic.
for n in nodes:
    n.write("flag", VersionedValue(value="green", version=1))
print("read ->", cluster.client_read("flag"))

# Simulate one node going dark — we still succeed because W=2 and we have 2 live nodes.
nodes[0].online = False
assert cluster.client_write("flag", "red")
print("one node offline, read ->", cluster.client_read("flag"))

nodes[0].online = True
print("back online, this stale node's local copy ->", nodes[0].store.get("flag"))


Notice the node that was offline still has the *old* value `green` — it missed the
`red` write. But a client reading with `R=2` will contact two nodes, at least one of
which has `red` (version is higher), so the client always sees `red`. The overlap rule
is doing its job.

## 4. Breaking the rule: `W + R ≤ N`

Let's pick `W=1, R=1` (very fast, very loose). Now a write can succeed on a single
node and a read can succeed on a single node, and those two nodes might be different.


In [ ]:
random.seed(7)
nodes = [Node(f"n{i}") for i in range(3)]
loose = QuorumCluster(N=3, W=1, R=1, nodes=nodes)

# To make the failure easy to see, only let the write land on one specific node.
nodes[1].online = False
nodes[2].online = False
assert loose.client_write("flag", "purple")  # only n0 has it
nodes[1].online = True
nodes[2].online = True

stale_reads = 0
fresh_reads = 0
for _ in range(200):
    got = loose.client_read("flag")
    if got is None:
        stale_reads += 1  # the one node with data wasn't the first one contacted
    elif got.value == "purple":
        fresh_reads += 1
    else:
        stale_reads += 1

print(f"fresh reads: {fresh_reads}  stale/missing reads: {stale_reads}")


About 2/3 of reads miss the freshly written value, because `R=1` happily returns the
first answer it gets — which is often from a node that doesn't know about the write yet.

With `W=1, R=1` you have essentially *no* consistency guarantee. That configuration is
still useful for caches or logging, but not for anything a user will complain about.

## 5. Exploring the whole table

Let's run a quick experiment across all reasonable combinations of (W, R) with N=3 and
see how often reads observe a just-completed write.


In [ ]:
def experiment(N: int, W: int, R: int, trials: int = 300) -> float:
    hits = 0
    for _ in range(trials):
        nodes = [Node(f"n{i}") for i in range(N)]
        cluster = QuorumCluster(N=N, W=W, R=R, nodes=nodes)
        if not cluster.client_write("k", "v1"):
            continue
        got = cluster.client_read("k")
        if got is not None and got.value == "v1":
            hits += 1
    return hits / trials


print(f"{'W':>2} {'R':>2} {'W+R':>4}  freshness")
for W in range(1, 4):
    for R in range(1, 4):
        freshness = experiment(N=3, W=W, R=R)
        print(f"{W:>2} {R:>2} {W+R:>4}  {freshness:.2%}")


You should see **100% freshness** whenever `W + R > N` (that is, `W + R >= 4`) and
lower freshness otherwise. That's the whole theorem, on your laptop.

## 6. The tradeoff

Bigger `W` or `R` give you stronger guarantees, but they cost:

| Tuning | Reads | Writes | Availability |
|---|---|---|---|
| Large `W` | fast | slow | writes fail when >`N−W` nodes are down |
| Large `R` | slow | fast | reads fail when >`N−R` nodes are down |
| `W + R > N` | strong consistency | slower | less forgiving of node failure |
| `W + R ≤ N` | stale reads possible | fast | very forgiving of node failure |

A common production config is **N=3, W=2, R=2** — strong enough to always see the
latest write, cheap enough that any single node can be down for maintenance.

### Sloppy quorums (a quick mention)

Real systems like Dynamo also support *sloppy quorums*: if the "correct" nodes for a
key are unreachable, writes are temporarily accepted by other nodes and later handed
off back to the right ones. That buys availability at the cost of occasionally serving
a stale read while the handoff completes. Good to know the term; the pure rule is what
you should remember.



## 7. Concurrent writes: the dark side of leaderless

Quorum reads see the latest write — but what if **two clients write at the exact same
time** to different subsets of nodes? There is no leader to serialize them. Both
writes succeed. Now what?

The simplest answer is **last-write-wins (LWW)**: every write carries a timestamp, and
the higher timestamp wins on conflict. That's exactly what our `version` field models.


In [ ]:
# Two concurrent writers updating the same key. Each writes to a different W=2 subset.
nodes = [Node(f"n{i}") for i in range(3)]
cluster = QuorumCluster(N=3, W=2, R=2, nodes=nodes)

# Manually craft two writes with version numbers we control to make the race obvious.
v_alice = VersionedValue(value="alice's value", version=10)
v_bob   = VersionedValue(value="bob's value",   version=11)  # bob's clock is slightly ahead

# Alice writes to n0, n1
nodes[0].write("key", v_alice)
nodes[1].write("key", v_alice)

# Bob writes to n1, n2 (overlap on n1!)
nodes[1].write("key", v_bob)
nodes[2].write("key", v_bob)

print("per-node state after the race:")
for n in nodes:
    print(" ", n.name, "->", n.store["key"])

print("client_read sees:", cluster.client_read("key"))


LWW resolved it cleanly: Bob's `version=11` beat Alice's `version=10`, and
`client_read` returns `bob's value`. Alice's write is silently *lost*.

That's the ugly truth about LWW: **a write that returned success can still be erased
by a concurrent write**. It's fast and simple, but if both writes were meaningful (e.g.
adding to a shopping cart), one user just got robbed.

Real systems mitigate this in three ways:

- **Vector clocks / version vectors** (Riak, classic Dynamo): instead of one number,
  every write tracks *which node bumped the version when*. The system can then *detect*
  conflicts and surface them to the application to merge.
- **CRDTs** (Redis Active-Active, Riak data types): data structures designed so
  concurrent writes always merge deterministically — counters, sets, and maps you can
  *never* lose updates from.
- **Single-leader-per-key** (Cassandra LWT, DynamoDB conditional writes): for the few
  operations that truly need it, fall back to a Paxos-style consensus round.

## 8. Read repair: how the lazy node catches up

In our experiments earlier, the offline node `n0` still held the **old** `green` value
after coming back online — even after we'd written `red`. In a real Dynamo-style
system, the *next read* through that node would notice it has a stale version and push
the newer value back. That's called **read repair**. Let's add it in 6 lines.


In [ ]:
class RepairingCluster(QuorumCluster):
    def client_read(self, key: str):
        picked = random.sample(self.nodes, self.R)
        answers = [(n, n.read(key)) for n in picked]
        non_null = [(n, a) for n, a in answers if a is not None]
        if not non_null:
            return None
        winner_node, winner = max(non_null, key=lambda na: na[1].version)
        # Push the winner back to any picked node that was behind.
        for n, a in answers:
            if a is None or a.version < winner.version:
                n.write(key, winner)
        return winner


nodes = [Node(f"n{i}") for i in range(3)]
cluster = RepairingCluster(N=3, W=2, R=2, nodes=nodes)

cluster.client_write("flag", "green")
nodes[0].online = False
cluster.client_write("flag", "red")  # n0 misses this
nodes[0].online = True

print("n0 BEFORE any read:", nodes[0].store["flag"])
# Force a read that includes n0 by picking R=N.
cluster.R = 3
cluster.client_read("flag")
print("n0 AFTER a read that included it:", nodes[0].store["flag"])


`n0` quietly upgraded itself to `red` because the read noticed a more-recent version
at another replica. Cassandra and DynamoDB do the same thing on every quorum read,
plus a periodic background sweep called **anti-entropy** (Merkle-tree comparisons
between nodes) to repair keys that nobody happens to read.

## 9. Multi-leader replication (a quick mention)

We've covered:

- **Single-leader** (notebooks 1 & 2) — one writer, many readers.
- **Leaderless** (this notebook) — every node accepts writes, conflicts via LWW/CRDTs.

There's a third flavour: **multi-leader replication** — multiple primaries, each
accepting writes, replicating to each other. You see it in:

- **CouchDB / Couchbase XDCR** — sync between data centers.
- **Active Directory / Cassandra DC-aware setups** — geo-distributed writes.
- **Calendar/notes apps that work offline** — your phone is its own "leader" until
  it syncs back.

It has the same conflict-resolution headaches as leaderless replication, plus extra
plumbing. People reach for it mainly when geographic write-locality matters more than
simplicity.

## 10. Real-world examples

| System | N / W / R defaults | Notable detail |
|---|---|---|
| **Cassandra** | N tunable, default `LOCAL_QUORUM` (W=R=majority) | Per-query consistency level — you choose tradeoff per call. |
| **Amazon DynamoDB** | N=3 across AZs, strong reads = quorum, eventual reads = single replica | The original Dynamo paper inspired this whole notebook. |
| **Riak** | N=3, W=2, R=2 | Pioneered vector clocks and sibling reconciliation. |
| **ScyllaDB** | Same model as Cassandra | Tuned for very high throughput on a single node. |
| **etcd / ZooKeeper / Consul** | Raft consensus, not Dynamo-style quorum | They look quorum-ish but actually use a single leader internally. |


## 11. Recap

- **Quorum replication = no leader, write to W nodes, read from R nodes, out of N total.**
- **Magic rule:** if `W + R > N`, reads see the latest successful write.
- Tuning (W, R) is the knob between consistency and availability — there is no free lunch.
- You just implemented, in a single notebook, the core algorithm behind Cassandra and
  DynamoDB.

Congrats — you've seen leader–follower, sync vs async, and quorums. That's the core
vocabulary of replication.
